In [3]:
import pandas as pd
import statsmodels.formula.api as smf
import scipy.stats as st
import matplotlib.pyplot as plt
from statsmodels.iolib.summary2 import summary_col
import numpy as np
from statsmodels.stats.diagnostic import het_breuschpagan

url = 'https://raw.githubusercontent.com/git-iush/large_family/refs/heads/main/data2.parquet'

In [4]:
df = pd.read_parquet(url)
df.head()

,SERIAL,HHWT,CLUSTER,STRATA,PERNUM,PERWT,NCHILD,SEX,AGE,MARST,...,BLACK,AIAN,API,OTHER,MULTR,HISP,EXPANSION,WELFARE,PRSNINC,YRSEDUC
0,4,4.0,2024000000041,280201,1,4.0,0,2,22,6,...,0,0,1,0,0,0,0,0.0,1000.0,13
1,9,74.0,2024000000091,280301,1,74.0,0,2,18,6,...,1,0,0,0,0,0,0,0.0,5100.0,12
2,12,55.0,2024000000121,160001,1,55.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,0.0,12
3,15,17.0,2024000000151,120201,1,17.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,6000.0,12
4,19,71.0,2024000000191,10001,1,71.0,0,2,23,6,...,0,0,0,0,0,0,0,0.0,0.0,13


In [5]:
medicaid_mapping = {1:0, 2:1}
df['MEDICAID'] = df['HINSCAID'].map(medicaid_mapping)
df['MEDFARE'] = df['MEDICAID'] * df['WELFARE']

df['FAMINC'] = df['FTOTINC'] - df['INCTOT']
df['PRSNINC'] = df['PRSNINC'] / 10000
df['FAMINC'] = df['FAMINC'] / 10000
df['PRSNINC2'] = df['PRSNINC'] ** 2
df['FAMINC2'] = df['FAMINC'] ** 2

df['AGE2'] = df['AGE'] ** 2

df['HIGHSCHOOL'] = [1 if i == 63 or i == 64 else 0 for i in df['EDUCD']]
df['BACHELORS'] = [1 if i == 101 else 0 for i in df['EDUCD']]
df['ADVANCED'] = [1 if i >= 114 else 0 for i in df['EDUCD']]

df['MARD'] = [1 if i == 1 or i == 2 or i == 3 else 0 for i in df['MARST']]

df['LARGEFAM'] = [1 if i >= 3 else 0 for i in df['NCHILD']]

In [6]:
ols_base = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + BLACK + AIAN + API + OTHER + MULTR + HISP + HIGHSCHOOL + BACHELORS + ADVANCED'

In [7]:
reg_base = smf.ols(ols_base, data=df).fit()
print(reg_base.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.317
Model:                            OLS   Adj. R-squared:                  0.317
Method:                 Least Squares   F-statistic:                 1.654e+04
Date:                Tue, 17 Nov 2026   Prob (F-statistic):               0.00
Time:                        13:21:13   Log-Likelihood:            -9.6634e+05
No. Observations:              678476   AIC:                         1.933e+06
Df Residuals:                  678456   BIC:                         1.933e+06
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.7656      0.014   -192.209      0.0

In [8]:
results = summary_col(
    [reg_base],
    model_names=['Baseline'],
    regressor_order=['MEDICAID', 'WELFARE', 'MEDFARE','AGE', 'AGE2', 'PRSNINC', 'PRSNINC2', 'FAMINC', 'FAMINC2','MARD', 'BLACK', 'AIAN', 'API', 'OTHER', 'MULTR', 'HISP','HIGHSCHOOL', 'BACHELORS', 'ADVANCED', 'Intercept'],
    stars=True,
    float_format='%0.5f',
    info_dict={'Observations': lambda x: f'{int(x.nobs):,}'},
    drop_omitted=True
)

print(results)


                 Baseline 
--------------------------
MEDICAID       0.31211*** 
               (0.00347)  
WELFARE        -0.32856***
               (0.02196)  
MEDFARE        0.61216*** 
               (0.02447)  
AGE            0.17769*** 
               (0.00093)  
AGE2           -0.00209***
               (0.00001)  
PRSNINC        -0.02813***
               (0.00041)  
PRSNINC2       0.00044*** 
               (0.00001)  
FAMINC         0.01285*** 
               (0.00022)  
FAMINC2        -0.00011***
               (0.00000)  
MARD           0.74744*** 
               (0.00299)  
BLACK          0.05794*** 
               (0.00456)  
AIAN           0.13044*** 
               (0.01035)  
API            -0.15332***
               (0.00462)  
OTHER          0.00391    
               (0.00605)  
MULTR          -0.02984***
               (0.00441)  
HISP           0.09236*** 
               (0.00432)  
HIGHSCHOOL     -0.08644***
               (0.00339)  
BACHELORS      -0.23818***


In [12]:
print(f'MEDICAID * WELFARE T TEST', reg_base.t_test("MEDFARE = 0"), sep='\n\n')

MEDICAID * WELFARE T TEST

                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.6122      0.024     25.018      0.000       0.564       0.660


In [13]:
print(f'MEDICAID + (MEDICAID * WELFARE) T TEST', reg_base.t_test("MEDICAID + MEDFARE = 0"), sep='\n\n')

MEDICAID + (MEDICAID * WELFARE) T TEST

                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.9243      0.024     38.126      0.000       0.877       0.972


In [14]:
print(f'WELFARE + (MEDICAID * WELFARE) T TEST', reg_base.t_test("WELFARE + MEDFARE = 0"), sep='\n\n')

WELFARE + (MEDICAID * WELFARE) T TEST

                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.2836      0.011     26.119      0.000       0.262       0.305


In [15]:
ftest = reg_base.f_test("MEDICAID = 0, WELFARE = 0, MEDFARE = 0")
print(f'JOINT SIGNIFICANCE F TEST', ftest, sep='\n\n')

JOINT SIGNIFICANCE F TEST

<F test: F=3578.290236364784, p=0.0, df_denom=6.78e+05, df_num=3>


In [16]:
df.shape

(720668, 88)